# CineFusion: CUR Recommender

This is the CUR notebook. It builds a CUR-decomposition collaborative filter on **MovieLens 25M**, blends it with the **TMDB 5000 BGE content embeddings**, and produces user recommendations, item-item similarity, and a hybrid recommender. Since Pradnya already did this for SVD, it makes sense we do it for CUR as well so we can evaluate them evenly.

### Pipeline
```
ml-25m/ratings.csv -> CUR pipeline -> User predictions --|
                              |                          |
                              |--> Item embeddings ------|    |--> Hybrid --> Top-N
                                                         |    |
tmdb-5000 + BGE embeddings --> Content scores ----------------|
```

### What we're doing
1. **Part 1**: Builds CUR on ml-25m via a checkpointed pipeline. Every step writes to disk and is skipped on re-run, so the 25M dataset can be processed in chunks across multiple sessions without recomputing everything. Big matrices use sparse `.npz` and small artifacts use CSV so they are human readable
2. **Part 2**: Evaluates CUR with **RMSE / MAE / Precision@10 / Recall@10 / HitRate@10 / NDCG@10**
3. **Part 3**: Item-item similarity from CUR factors (no full reconstruction)
4. **Part 4**: Bridges ml-25m and tmdb-5000 via `links.csv` and loads the
   precomputed BGE embeddings (thanks Harsh)
5. **Part 5**: Hybrid recommender that blends CUR (collaborative) with BGE (content) using a single tunable weight α, plus the same comparisons and α-sensitivity sweep used in the Pradnya's SVD notebook
6. **Part 6**: Summary table

### Reads from Drive
* `ml-25m/ratings.csv`
* `ml-25m/movies.csv`
* `ml-25m/links.csv`
* `tmdb-5000/tmdb_5000_movies.csv`
* `tmdb-5000-embeddings-BAAI/embeddings.npy`

### Writes to Drive
* `cur-output/cur_checkpoints/`: all intermediate CUR artifacts
* `cur-output/cur_user_recs.csv`: top-10 CUR predictions per user
* `cur-output/cur_hybrid_sample.csv`: sample hybrid output for the report

## Step 0: Setup
Configuring paths imports as well as global variables.

In [ ]:
# For Colab:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/CineFusion"

#-------------------------------------------------

# For local:
# BASE_DIR = "."

#-------------------------------------------------

import os, json, gc
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix, save_npz, load_npz
from numpy.linalg import svd

# ----------------- CONFIG -----------------
RATINGS_PATH       = f"{BASE_DIR}/ml-25m/ratings.csv"
MOVIES_PATH        = f"{BASE_DIR}/ml-25m/movies.csv"
LINKS_PATH         = f"{BASE_DIR}/ml-25m/links.csv"
TMDB_MOVIES_PATH   = f"{BASE_DIR}/tmdb-5000/tmdb_5000_movies.csv"
TMDB_CREDITS_PATH  = f"{BASE_DIR}/tmdb-5000/tmdb_5000_credits.csv"
BGE_EMBED_PATH     = f"{BASE_DIR}/tmdb-5000-embeddings-BAAI/embeddings.npy"

OUTPUT_DIR         = f"{BASE_DIR}/cur-output"
CHECKPOINT_DIR     = f"{OUTPUT_DIR}/cur_checkpoints"

SAMPLE_FRACTION    = 1.0
TEST_FRACTION      = 0.2
R_FINAL            = 500     # CUR sketch size
MIN_SUPPORT        = 20      # min training ratings for a movie to be recommendable
ALPHA              = 0.6     # hybrid weight: ALPHA * CUR + (1-ALPHA) * CBF
SEED               = 42
# ------------------------------------------

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
RNG = np.random.default_rng(SEED)

def ckpt(name): return os.path.join(CHECKPOINT_DIR, name)
def done(name): return os.path.exists(ckpt(name))
def atomic_write_csv(df, name):
    tmp = ckpt(name + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, ckpt(name))

print("Checkpoint dir :", os.path.abspath(CHECKPOINT_DIR))
print("Existing files :", sorted(os.listdir(CHECKPOINT_DIR)) or "(empty)")

Checkpoint dir : c:\Users\Justin\Desktop\temp\cinedata\cur-output\cur_checkpoints
Existing files : (empty)


---
## Part 1: CUR Pipeline

Every step writes its outputs to the output folder and is guarded by a `done(...)` check.

The choice of file format matters for the 25M dataset:

| Artifact | Shape (ml-25m) | Format | Why |
|---|---|---|---|
| User / movie indices, means, norms, samples | small | **CSV** | Open in Excel, diff in git |
| Centered user-item matrix | 162k × 62k sparse | **.npz** | CSV would be ~600 MB and take too long to parse |
| C, R | sparse | **.npz** | Same as above |
| W, U | 500 × 500 dense | **CSV** | Small enough, inspectable |
| Test set | ~5M rows | **.npz** | Tiny in npz (30+gb memory otherwise), slow as CSV |

### 1.1: Load ratings, train/test split, build sparse centered matrix

In [ ]:
STEP_OUTPUTS_1 = ["user_index.csv", "movie_index.csv", "user_means.csv",
                  "M_centered.npz", "test_set.npz", "meta.json"]

if all(done(f) for f in STEP_OUTPUTS_1):
    print("Step 1.1 already complete — skipping.")
else:
    print(f"Step 1.1: reading {RATINGS_PATH}")
    ratings = pd.read_csv(RATINGS_PATH, usecols=["userId", "movieId", "rating"])
    print(f"  loaded {len(ratings):,} ratings")

    if SAMPLE_FRACTION < 1.0:
        ratings = ratings.sample(frac=SAMPLE_FRACTION, random_state=SEED)
        print(f"  sampled to {len(ratings):,} ({SAMPLE_FRACTION:.0%})")

    # Stable indices
    user_ids  = np.sort(ratings["userId"].unique())
    movie_ids = np.sort(ratings["movieId"].unique())
    u2i = {u: i for i, u in enumerate(user_ids)}
    m2i = {m: i for i, m in enumerate(movie_ids)}
    n_users, n_movies = len(user_ids), len(movie_ids)
    print(f"  users: {n_users:,}   movies: {n_movies:,}")
    print(f"  sparsity: {1 - len(ratings)/(n_users*n_movies):.4%}")

    ratings["uidx"] = ratings["userId"].map(u2i).astype(np.int32)
    ratings["midx"] = ratings["movieId"].map(m2i).astype(np.int32)

    # Train / test split (random row-wise)
    rng_split = np.random.default_rng(SEED)
    is_test = rng_split.random(len(ratings)) < TEST_FRACTION
    train = ratings[~is_test]
    test  = ratings[is_test]
    print(f"  train: {len(train):,}   test: {len(test):,}")

    # Per-user mean, computed on TRAIN only
    user_means = train.groupby("uidx")["rating"].agg(["mean", "count"]).reset_index()
    user_means.columns = ["uidx", "mean", "n_ratings"]
    mean_lookup = user_means.set_index("uidx")["mean"]
    global_mean = float(train["rating"].mean())

    # Center training ratings (any user missing from train falls back to global mean)
    train_means = train["uidx"].map(mean_lookup).fillna(global_mean).astype(np.float32)
    train_centered = (train["rating"].astype(np.float32) - train_means).values

    M_centered = csr_matrix(
        (train_centered, (train["uidx"].values, train["midx"].values)),
        shape=(n_users, n_movies), dtype=np.float32,
    )
    print(f"  sparse training matrix: {M_centered.shape}, nnz={M_centered.nnz:,}")

    # Save everything
    atomic_write_csv(pd.DataFrame({"userId": user_ids, "uidx": np.arange(n_users)}),
                     "user_index.csv")
    atomic_write_csv(pd.DataFrame({"movieId": movie_ids, "midx": np.arange(n_movies)}),
                     "movie_index.csv")
    atomic_write_csv(user_means, "user_means.csv")
    save_npz(ckpt("M_centered.npz"), M_centered)

    # Test set as compact npz, not CSV (~5M rows on full 25m)
    np.savez_compressed(
        ckpt("test_set.npz"),
        uidx=test["uidx"].values.astype(np.int32),
        midx=test["midx"].values.astype(np.int32),
        rating=test["rating"].values.astype(np.float32),
    )

    with open(ckpt("meta.json"), "w") as f:
        json.dump({"n_users": int(n_users), "n_movies": int(n_movies),
                    "n_train": int(len(train)), "n_test": int(len(test)),
                    "global_mean": global_mean, "sample_fraction": SAMPLE_FRACTION,
                    "seed": SEED}, f)

    del ratings, train, test, M_centered
    gc.collect()
    print("Step 1.1 done.")

Step 1.1: reading ./ml-25m/ratings.csv
  loaded 25,000,095 ratings
  users: 162,541   movies: 59,047
  sparsity: 99.7395%
  train: 20,001,207   test: 4,998,888
  sparse training matrix: (162541, 59047), nnz=20,001,207
Step 1.1 done.


### 1.2: Frobenius column / row probabilities

In [ ]:
STEP_OUTPUTS_2 = ["col_norms.csv", "row_norms.csv", "fro.json"]

if all(done(f) for f in STEP_OUTPUTS_2):
    print("Step 1.2 already complete — skipping.")
else:
    M = load_npz(ckpt("M_centered.npz"))
    sq = M.multiply(M)
    col_norms_sq = np.asarray(sq.sum(axis=0)).ravel()
    row_norms_sq = np.asarray(sq.sum(axis=1)).ravel()
    fro_sq = float(col_norms_sq.sum())
    print(f"  ||M||_F^2 = {fro_sq:.4f}")

    p_col = col_norms_sq / fro_sq
    p_row = row_norms_sq / fro_sq

    atomic_write_csv(pd.DataFrame({"midx": np.arange(len(col_norms_sq)),
                                    "norm_sq": col_norms_sq, "prob": p_col}),
                     "col_norms.csv")
    atomic_write_csv(pd.DataFrame({"uidx": np.arange(len(row_norms_sq)),
                                    "norm_sq": row_norms_sq, "prob": p_row}),
                     "row_norms.csv")
    with open(ckpt("fro.json"), "w") as f:
        json.dump({"fro_sq": fro_sq}, f)

    del M, sq
    print("Step 1.2 done.")

  ||M||_F^2 = 17895702.0000
Step 1.2 done.


### 1.3: Sample r columns and r rows

In [ ]:
STEP_OUTPUTS_3 = ["sampled_cols.csv", "sampled_rows.csv"]

if all(done(f) for f in STEP_OUTPUTS_3):
    print("Step 1.3 already complete — skipping.")
else:
    col_norms = pd.read_csv(ckpt("col_norms.csv"))
    row_norms = pd.read_csv(ckpt("row_norms.csv"))

    # Renormalize after CSV round-trip
    p_col = col_norms["prob"].values.astype(np.float64); p_col /= p_col.sum()
    p_row = row_norms["prob"].values.astype(np.float64); p_row /= p_row.sum()

    rng = np.random.default_rng(SEED)
    col_pick = rng.choice(len(col_norms), size=R_FINAL, replace=True, p=p_col)
    row_pick = rng.choice(len(row_norms), size=R_FINAL, replace=True, p=p_row)

    atomic_write_csv(pd.DataFrame({
        "col_pos": np.arange(R_FINAL), "midx": col_pick,
        "prob": p_col[col_pick], "scale": 1.0/np.sqrt(R_FINAL * p_col[col_pick])}),
        "sampled_cols.csv")
    atomic_write_csv(pd.DataFrame({
        "row_pos": np.arange(R_FINAL), "uidx": row_pick,
        "prob": p_row[row_pick], "scale": 1.0/np.sqrt(R_FINAL * p_row[row_pick])}),
        "sampled_rows.csv")
    print(f"  sampled {R_FINAL} cols and {R_FINAL} rows")
    print("Step 1.3 done.")

  sampled 500 cols and 500 rows
Step 1.3 done.


### 1.4: Build sparse C

In [ ]:
if done("C.npz"):
    print("Step 1.4 already complete — skipping.")
else:
    M = load_npz(ckpt("M_centered.npz"))
    sc = pd.read_csv(ckpt("sampled_cols.csv"))
    C = M[:, sc["midx"].values].astype(np.float32)
    C = (C @ sp.diags(sc["scale"].values.astype(np.float32))).tocsr()
    print(f"  C: shape={C.shape}, nnz={C.nnz:,}")
    save_npz(ckpt("C.npz"), C)
    del M, C
    print("Step 1.4 done.")

  C: shape=(162541, 500), nnz=5,775,028
Step 1.4 done.


### 1.5: Build sparse R

In [ ]:
if done("R.npz"):
    print("Step 1.5 already complete — skipping.")
else:
    M = load_npz(ckpt("M_centered.npz"))
    sr = pd.read_csv(ckpt("sampled_rows.csv"))
    R_mat = M[sr["uidx"].values, :].astype(np.float32)
    R_mat = (sp.diags(sr["scale"].values.astype(np.float32)) @ R_mat).tocsr()
    print(f"  R: shape={R_mat.shape}, nnz={R_mat.nnz:,}")
    save_npz(ckpt("R.npz"), R_mat)
    del M, R_mat
    print("Step 1.5 done.")

  R: shape=(500, 59047), nnz=228,598
Step 1.5 done.


### 1.6: Build W and U

`W` is the `R_FINAL x R_FINAL` intersection of the sampled rows and columns, scaled by both. `U = pinv(W)` via SVD with the singular values inverted and squared (CUR formula).

In [ ]:
STEP_OUTPUTS_6 = ["W.csv", "U.csv"]

if all(done(f) for f in STEP_OUTPUTS_6):
    print("Step 1.6 already complete — skipping.")
else:
    M  = load_npz(ckpt("M_centered.npz"))
    sc = pd.read_csv(ckpt("sampled_cols.csv"))
    sr = pd.read_csv(ckpt("sampled_rows.csv"))

    W = np.asarray(M[sr["uidx"].values, :][:, sc["midx"].values].todense(),
                   dtype=np.float32)
    col_scale = sc["scale"].values.astype(np.float32)
    row_scale = sr["scale"].values.astype(np.float32)
    W = W * col_scale[None, :] * row_scale[:, None]

    X, sigma, Yt = svd(W, full_matrices=False)
    sigma_inv = np.where(sigma > 1e-10, 1.0 / sigma, 0.0)
    U = (Yt.T * (sigma_inv ** 2)) @ X.T
    print(f"  W: {W.shape}    cond ≈ {(sigma.max()/max(sigma.min(),1e-12)):.2e}")
    print(f"  effective rank (sigma > 1e-6): {(sigma > 1e-6).sum()} / {len(sigma)}")

    rr, cc = np.meshgrid(np.arange(W.shape[0]), np.arange(W.shape[1]), indexing="ij")
    atomic_write_csv(pd.DataFrame({"row": rr.ravel(), "col": cc.ravel(), "value": W.ravel()}),
                     "W.csv")
    atomic_write_csv(pd.DataFrame({"row": rr.ravel(), "col": cc.ravel(), "value": U.ravel()}),
                     "U.csv")
    del M, W, U, X, Yt
    print("Step 1.6 done.")

  W: (500, 500)    cond ≈ 8.39e+14
  effective rank (sigma > 1e-6): 428 / 500
Step 1.6 done.


---
## Part 2: Evaluate CUR

### 2.1: Load model artifacts

Loads the sparse factors, the dense U, the index files, and pre-computes `UR` so that prediction for any user is a length-`R_FINAL` dot product.

In [ ]:
C     = load_npz(ckpt("C.npz"))
R_mat = load_npz(ckpt("R.npz"))

U_df = pd.read_csv(ckpt("U.csv"))
U    = U_df.pivot(index="row", columns="col", values="value").values.astype(np.float32)

user_index  = pd.read_csv(ckpt("user_index.csv"))
movie_index = pd.read_csv(ckpt("movie_index.csv"))
user_means  = pd.read_csv(ckpt("user_means.csv"))
with open(ckpt("meta.json")) as f: META = json.load(f)

userId_to_uidx  = dict(zip(user_index.userId,  user_index.uidx))
midx_to_movieId = dict(zip(movie_index.midx,    movie_index.movieId))
uidx_to_mean    = dict(zip(user_means.uidx,    user_means["mean"]))
GLOBAL_MEAN     = META["global_mean"]

n_users  = META["n_users"]
n_movies = META["n_movies"]

# Pre-compute U @ R
UR = (U @ R_mat).astype(np.float32)
print(f"C={C.shape}  U={U.shape}  R={R_mat.shape}  UR={UR.shape}")

# Support filter: count of training ratings per movie (column nnz of M_centered)
M_train = load_npz(ckpt("M_centered.npz"))
movie_support = np.asarray((M_train != 0).sum(axis=0)).ravel()
eligible_mask_score = np.where(movie_support >= MIN_SUPPORT, 0.0, -np.inf).astype(np.float32)
print(f"Eligible movies (support >= {MIN_SUPPORT}): {(movie_support >= MIN_SUPPORT).sum():,}")

C=(162541, 500)  U=(500, 500)  R=(500, 59047)  UR=(500, 59047)
Eligible movies (support >= 20): 16,967


In [ ]:
# Vectorized score row for one user with ratings clipped to MovieLens range
def predict_user_row(uidx):
    c_row = np.asarray(C[uidx, :].todense()).ravel()
    return c_row @ UR + uidx_to_mean.get(uidx, GLOBAL_MEAN)

def predict_user_clipped(uidx):
    return np.clip(predict_user_row(uidx), 0.5, 5.0)

### 2.2: RMSE & MAE on held-out test set

In [ ]:
test_set = np.load(ckpt("test_set.npz"))
test_uidx, test_midx, test_rating = test_set["uidx"], test_set["midx"], test_set["rating"]
print(f"Test set: {len(test_rating):,} ratings")

# Predict in batches grouped by uidx so we only compute one prediction row per user
order = np.argsort(test_uidx, kind="stable")
test_uidx_s = test_uidx[order]; test_midx_s = test_midx[order]; test_rating_s = test_rating[order]

preds = np.empty(len(test_rating_s), dtype=np.float32)
i = 0
unique_users, starts = np.unique(test_uidx_s, return_index=True)
starts = np.append(starts, len(test_rating_s))
for k, u in enumerate(unique_users):
    s, e = starts[k], starts[k+1]
    row = predict_user_clipped(int(u))
    preds[s:e] = row[test_midx_s[s:e]]

errors  = preds - test_rating_s
RMSE    = float(np.sqrt(np.mean(errors ** 2)))
MAE     = float(np.mean(np.abs(errors)))
print(f"RMSE: {RMSE:.4f}")
print(f"MAE : {MAE:.4f}")

Test set: 4,998,888 ratings
RMSE: 0.9569
MAE : 0.7399


### 2.3: Precision@10 and Recall@10 (with support filter)

Same evaluation as in our intermediate report: a held-out rating >= 4 counts as relevant, the top 10 unseen + eligible (support >= `MIN_SUPPORT`) movies count as the recommendation set, and we average over users with at least 2 relevant test items.

In [ ]:
from collections import defaultdict

K, RELEVANCE_THRESHOLD, MIN_RELEVANT = 10, 4.0, 2

# Build per-user "seen in training" sets from M_train (any nonzero is seen)
M_train_csr = M_train.tocsr()
def seen_midx_for(uidx):
    return M_train_csr[uidx, :].indices

#----------------------------
# Precision@10 / Recall@10

relevant = defaultdict(set)
for u, m, r in zip(test_uidx, test_midx, test_rating):
    if r >= RELEVANCE_THRESHOLD:
        relevant[int(u)].add(int(m))

precisions, recalls, n_eval = [], [], 0
for u, rel in relevant.items():
    if len(rel) < MIN_RELEVANT: continue
    scores = predict_user_clipped(u) + eligible_mask_score
    scores[seen_midx_for(u)] = -np.inf
    top_k = np.argpartition(-scores, K)[:K]
    hits  = len(set(top_k.tolist()) & rel)
    precisions.append(hits / K)
    recalls.append(hits / len(rel))
    n_eval += 1

P_at_10 = float(np.mean(precisions))
R_at_10 = float(np.mean(recalls))

# --------------------------
# HitRate@10 and NDCG@10:
# For each held-out positive (rating >= threshold), sample 99 unseen negatives, rank the 100 candidates
# by predicted score, and ask whether the positive is in the top 10. NDCG gives partial credit by rank position.

NUM_NEGATIVES = 99
rng_eval = np.random.default_rng(SEED)

# Group test positives per user for batched scoring
user_to_test_positives = defaultdict(list)
for u, m, r in zip(test_uidx, test_midx, test_rating):
    if r >= RELEVANCE_THRESHOLD:
        user_to_test_positives[int(u)].append(int(m))

# Precompute the eligible movie pool (support >= MIN_SUPPORT)
eligible_pool = np.where(eligible_mask_score == 0.0)[0]

hits, ndcgs, n_samples = [], [], 0
log2_positions = 1.0 / np.log2(np.arange(2, K + 2))   # discount factors

for u, positives in user_to_test_positives.items():
    if len(positives) < MIN_RELEVANT: continue
    seen = set(seen_midx_for(u).tolist()) | set(positives)
    score_row = predict_user_clipped(u)

    for pos in positives:
        # Sample 99 negatives from eligible pool, excluding seen items
        negs, attempts = [], 0
        while len(negs) < NUM_NEGATIVES and attempts < NUM_NEGATIVES * 5:
            cand = rng_eval.choice(eligible_pool, size=NUM_NEGATIVES * 2)
            negs.extend(int(x) for x in cand if int(x) not in seen)
            attempts += 1
        negs = negs[:NUM_NEGATIVES]
        if len(negs) < NUM_NEGATIVES: continue

        candidates = np.array([pos] + negs, dtype=np.int64)
        cand_scores = score_row[candidates]
        # Rank position of the positive (0-indexed)
        rank = (cand_scores > cand_scores[0]).sum()

        if rank < K:
            hits.append(1.0)
            ndcgs.append(log2_positions[rank])
        else:
            hits.append(0.0)
            ndcgs.append(0.0)
        n_samples += 1

HitRate_at_10 = float(np.mean(hits))
NDCG_at_10    = float(np.mean(ndcgs))

print(f"--- Strict full-ranking metrics ---")
print(f"Evaluated on {n_eval:,} users")
print(f"Precision@10 : {P_at_10:.4f}")
print(f"Recall@10    : {R_at_10:.4f}")
print()
print(f"--- Sampled metrics ---")
print(f"Evaluated on {n_samples:,} (user, positive) pairs")
print(f"HitRate@10   : {HitRate_at_10:.4f}")
print(f"NDCG@10      : {NDCG_at_10:.4f}")


--- Strict full-ranking metrics ---
Evaluated on 150,532 users
Precision@10 : 0.0232
Recall@10    : 0.0165

--- Sampled metrics ---
Evaluated on 2,482,278 (user, positive) pairs
HitRate@10   : 0.2805
NDCG@10      : 0.1720


### 2.4: Save per-user top-10 CUR recommendations

In [ ]:
# We materialize the table in chunks of users to keep memory bounded.
OUT_PATH = f"{OUTPUT_DIR}/cur_user_recs.csv"
TOP_N = 10
CHUNK = 5000

if os.path.exists(OUT_PATH):
    print(f"  {OUT_PATH} already exists — skipping. Delete it to rebuild.")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    rows = []
    uidx_to_userId = dict(zip(user_index.uidx, user_index.userId))
    for start in range(0, n_users, CHUNK):
        end = min(start + CHUNK, n_users)
        for u in range(start, end):
            scores = predict_user_clipped(u) + eligible_mask_score
            scores[seen_midx_for(u)] = -np.inf
            top = np.argpartition(-scores, TOP_N)[:TOP_N]
            top = top[np.argsort(-scores[top])]
            for rank, midx in enumerate(top):
                rows.append((int(uidx_to_userId[u]),
                             int(midx_to_movieId[int(midx)]),
                             float(scores[midx]),
                             rank + 1))
        if (end // CHUNK) % 5 == 0:
            print(f"  ... {end:,}/{n_users:,} users")
    user_recs_df = pd.DataFrame(rows, columns=["userId", "movieId", "cur_score", "rank"])
    user_recs_df.to_csv(OUT_PATH, index=False)
    print(f"  Saved {len(user_recs_df):,} rows to {OUT_PATH}")
    del rows, user_recs_df
    gc.collect()

  ... 25,000/162,541 users
  ... 50,000/162,541 users
  ... 75,000/162,541 users
  ... 100,000/162,541 users
  ... 125,000/162,541 users
  ... 150,000/162,541 users
  Saved 1,625,410 rows to ./cur-output/cur_user_recs.csv


---
## Part 3: Item-Item Similarity from CUR

Cosine similarity between movie columns of the CUR reconstruction.

In [ ]:
if done("item_embeddings.npz"):
    print("Step 3 already complete — skipping.")
else:
    A = (C @ U).astype(np.float32)              # (n_users, R_FINAL)
    G = A.T @ A
    G = (G + G.T) / 2                           # numerical hygiene
    sigma_g, V_g = np.linalg.eigh(G)
    sigma_g = np.clip(sigma_g, 0.0, None)
    L = (np.sqrt(sigma_g)[:, None] * V_g.T).astype(np.float32)
    E = (L @ R_mat).astype(np.float32)          # (R_FINAL, n_movies)
    print(f"  E={E.shape}   effective rank={(sigma_g > 1e-6).sum()}/{len(sigma_g)}")
    np.savez_compressed(ckpt("item_embeddings.npz"), E=E)
    del A, G, V_g, L, E
    print("Step 3 done.")

  E=(500, 59047)   effective rank=427/500
Step 3 done.


In [ ]:
# Load embeddings + helpers
_e = np.load(ckpt("item_embeddings.npz"))
E = _e["E"]
E_norms = np.linalg.norm(E, axis=0) + 1e-12

movies = pd.read_csv(MOVIES_PATH)
title_lookup    = dict(zip(movies.movieId, movies.title))
movieId_to_midx = dict(zip(movie_index.movieId, movie_index.midx))

_norm_str = lambda s: "".join(ch.lower() for ch in str(s) if ch.isalnum())
title_to_movieId = {_norm_str(t): mid for mid, t in title_lookup.items()}
eligible_bool = (movie_support >= MIN_SUPPORT)

def _resolve_movie(query):
    if isinstance(query, (int, np.integer)):
        if query not in movieId_to_midx:
            raise KeyError(f"movieId {query} not in index")
        return movieId_to_midx[query]
    key = _norm_str(query)
    if key in title_to_movieId and title_to_movieId[key] in movieId_to_midx:
        return movieId_to_midx[title_to_movieId[key]]
    matches = [(mid, t) for mid, t in title_lookup.items()
               if key in _norm_str(t) and mid in movieId_to_midx]
    if not matches:
        raise KeyError(f"No movie matching {query!r}")
    matches.sort(key=lambda x: movie_support[movieId_to_midx[x[0]]], reverse=True)
    if len(matches) > 1:
        print(f"  ambiguous -- using {matches[0][1]!r}")
    return movieId_to_midx[matches[0][0]]

def similar_movies(query, top_k=10):
    j = _resolve_movie(query)
    sims = (E[:, j] @ E) / (E_norms[j] * E_norms)
    sims[~eligible_bool] = -np.inf
    sims[j] = -np.inf
    top = np.argpartition(-sims, top_k)[:top_k]
    top = top[np.argsort(-sims[top])]
    return pd.DataFrame({
        "movieId":    [midx_to_movieId[int(i)] for i in top],
        "title":      [title_lookup.get(midx_to_movieId[int(i)], "?") for i in top],
        "similarity": [round(float(sims[i]), 4) for i in top],
        "support":    [int(movie_support[i]) for i in top],
    })

similar_movies("Toy Story (1995)", top_k=10)

,movieId,title,similarity,support
0,106540,Delivery Man (2013),0.8828,301
1,32799,Maidens in Uniform (Mädchen in Uniform) (1931),0.8805,23
2,85020,"Mechanic, The (2011)",0.8795,1481
3,135534,Krampus (2015),0.8722,317
4,5440,She Wore a Yellow Ribbon (1949),0.8704,315
5,45508,Wah-Wah (2005),0.8703,24
6,26974,Gummo (1997),0.8697,405
7,7353,Clifford (1994),0.8681,111
8,26,Othello (1995),0.8677,2049
9,2720,Inspector Gadget (1999),0.8665,3313


---
## Part 4: TMDB Content Bridge

We need to:
1. Load the precomputed BGE embeddings produced by Harsh
2. Bridge ml-25m `movieId` <-> tmdb-5000 `id` via `links.csv` so we can convert between collaborative and content space

In [ ]:
# 4.1 -- Load TMDB metadata

tmdb_movies_raw = pd.read_csv(TMDB_MOVIES_PATH)
credits = pd.read_csv(f"{BASE_DIR}/tmdb-5000/tmdb_5000_credits.csv")
tmdb_movies = tmdb_movies_raw.merge(credits, on="title")
print(f"TMDB movies after merge with credits: {len(tmdb_movies):,}")

# 4.2 -- Load BGE embeddings
if not os.path.exists(BGE_EMBED_PATH):
    raise FileNotFoundError(
        f"BGE embeddings not found at {BGE_EMBED_PATH}. "
        f"Run the content-based notebook (CineFusion_CF.ipynb) first to "
        f"generate `tmdb-5000-embeddings-BAAI/embeddings.npy`."
    )
bge_embeddings = np.load(BGE_EMBED_PATH).astype(np.float32)
bge_embeddings /= (np.linalg.norm(bge_embeddings, axis=1, keepdims=True) + 1e-12)
print(f"BGE embeddings: {bge_embeddings.shape}")

if len(bge_embeddings) != len(tmdb_movies):
    print(f"  WARNING: embeddings ({len(bge_embeddings)}) != merged tmdb ({len(tmdb_movies)})")
    print(f"  Truncating to the shorter of the two.")
    n = min(len(bge_embeddings), len(tmdb_movies))
    bge_embeddings = bge_embeddings[:n]
    tmdb_movies = tmdb_movies.iloc[:n].reset_index(drop=True)

# Build tmdb_id -> row in the embeddings matrix
tmdbId_to_tmdbidx = dict(zip(tmdb_movies["id"].astype(int), np.arange(len(tmdb_movies))))

TMDB movies after merge with credits: 4,809
BGE embeddings: (4809, 768)


In [ ]:
# 4.3 -- Bridge ml-25m movieId <-> tmdb tmdb_id via links.csv
links = pd.read_csv(LINKS_PATH).dropna(subset=["tmdbId"])
links["tmdbId"] = links["tmdbId"].astype(int)

bridge = (links
          .merge(movie_index, on="movieId", how="inner")        # adds midx
          [["movieId", "midx", "tmdbId"]])
bridge["tmdb_idx"] = bridge["tmdbId"].map(tmdbId_to_tmdbidx)
bridge = bridge.dropna(subset=["tmdb_idx"])
bridge["tmdb_idx"] = bridge["tmdb_idx"].astype(int)

print(f"Linked movies: {len(bridge):,} of {n_movies:,} ml-25m movies "
      f"({len(bridge)/n_movies:.1%})")

midx_to_tmdb_idx = dict(zip(bridge.midx, bridge.tmdb_idx))
tmdb_idx_to_midx = dict(zip(bridge.tmdb_idx, bridge.midx))

Linked movies: 4,595 of 59,047 ml-25m movies (7.8%)


---
## Part 5: Hybrid Recommender

Same blending strategy as the ALS notebook, with CUR replacing ALS as the
collaborative signal:

$$\text{hybrid}(u, m) \;=\; \alpha \cdot \text{cur\_norm}(u, m) \;+\;
                       (1-\alpha) \cdot \text{cbf}(\text{seed}, m)$$

where `cur_norm` is the user's CUR predicted rating min-max normalized across their candidate set, and `cbf` is the BGE cosine similarity between the seed movie and the candidate.

In [ ]:
def _normalize(arr):
    lo, hi = arr.min(), arr.max()
    if hi - lo < 1e-9: return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

def get_cur_scores(user_id, candidate_midx):
    """Predicted CUR rating for one user across a set of midx candidates."""
    if user_id not in userId_to_uidx: return None
    uidx = userId_to_uidx[user_id]
    row = predict_user_clipped(uidx)
    return row[candidate_midx]

def get_cbf_scores(seed_movie, candidate_midx):
    """BGE cosine similarity between seed and each candidate (same length array,
    -inf for candidates not in the TMDB bridge)."""
    seed_midx = _resolve_movie(seed_movie)
    if seed_midx not in midx_to_tmdb_idx:
        raise ValueError(f"Seed movie has no TMDB embedding "
                         f"(not in links.csv or not in tmdb-5000)")
    seed_vec = bge_embeddings[midx_to_tmdb_idx[seed_midx]]

    out = np.full(len(candidate_midx), -np.inf, dtype=np.float32)
    for i, m in enumerate(candidate_midx):
        if m in midx_to_tmdb_idx:
            out[i] = float(seed_vec @ bge_embeddings[midx_to_tmdb_idx[m]])
    return out

def hybrid_recommend(user_id, seed_movie, alpha=ALPHA, top_n=10, candidate_pool=200):
    """
    1. Pull the user's top `candidate_pool` CUR predictions (eligible, unseen).
    2. Score each candidate with BGE cosine vs the seed movie.
    3. Normalize both signals to [0, 1] and blend with weight alpha.
    """
    if user_id not in userId_to_uidx:
        print(f"User {user_id} not in training data."); return None
    uidx = userId_to_uidx[user_id]

    # Step 1: candidate pool from CUR
    scores = predict_user_clipped(uidx) + eligible_mask_score
    scores[seen_midx_for(uidx)] = -np.inf
    pool_idx = np.argpartition(-scores, candidate_pool)[:candidate_pool]
    pool_idx = pool_idx[np.isfinite(scores[pool_idx])]
    cur_raw = scores[pool_idx]

    # Step 2: CBF scores
    cbf_raw = get_cbf_scores(seed_movie, pool_idx)
    finite = np.isfinite(cbf_raw)
    if finite.sum() == 0:
        print("No candidates have BGE embeddings -- returning CUR-only ranking.")
        finite = np.ones_like(cbf_raw, dtype=bool)
        cbf_raw = np.zeros_like(cbf_raw)

    # Step 3: normalize and blend (only on candidates with both signals)
    pool_idx, cur_raw, cbf_raw = pool_idx[finite], cur_raw[finite], cbf_raw[finite]
    cur_norm = _normalize(cur_raw)
    cbf_norm = _normalize(cbf_raw)
    hybrid   = alpha * cur_norm + (1 - alpha) * cbf_norm

    order = np.argsort(-hybrid)[:top_n]
    return pd.DataFrame({
        "title":         [title_lookup.get(midx_to_movieId[int(pool_idx[i])], "?") for i in order],
        "movieId":       [int(midx_to_movieId[int(pool_idx[i])]) for i in order],
        "cur_norm":      [round(float(cur_norm[i]), 3) for i in order],
        "cbf_score":     [round(float(cbf_norm[i]), 3) for i in order],
        "hybrid_score":  [round(float(hybrid[i]),   3) for i in order],
    })

### 5.1: Hybrid on a sample user

In [ ]:
SAMPLE_USER = 1
SEED_MOVIE  = "Toy Story (1995)"

print(f"Hybrid Recommendations for User {SAMPLE_USER}  (seed: {SEED_MOVIE!r}, alpha={ALPHA})")
print("=" * 75)
hybrid_recommend(SAMPLE_USER, SEED_MOVIE, alpha=ALPHA, top_n=10)

Hybrid Recommendations for User 1  (seed: 'Toy Story (1995)', alpha=0.6)


,title,movieId,cur_norm,cbf_score,hybrid_score
0,Toy Story 3 (2010),78499,0.686,0.992,0.808
1,Toy Story 2 (1999),3114,0.532,1.000,0.719
2,Spirited Away (Sen to Chihiro no kamikakushi) ...,5618,1.000,0.239,0.696
3,Up (2009),68954,0.786,0.549,0.691
4,"Piano, The (1993)",509,0.729,0.287,0.553
5,Psycho (1960),1219,0.769,0.179,0.533
6,Die Hard 2 (1990),1370,0.759,0.150,0.515
7,That Thing You Do! (1996),1042,0.571,0.357,0.485
8,Annie Hall (1977),1230,0.435,0.529,0.472
9,Harry Potter and the Sorcerer's Stone (a.k.a. ...,4896,0.462,0.468,0.464


### 5.2: Compare CUR-only vs CBF-only vs Hybrid

In [ ]:
def cur_only_top(user_id, top_n=10):
    uidx = userId_to_uidx[user_id]
    scores = predict_user_clipped(uidx) + eligible_mask_score
    scores[seen_midx_for(uidx)] = -np.inf
    top = np.argpartition(-scores, top_n)[:top_n]
    top = top[np.argsort(-scores[top])]
    return [title_lookup.get(midx_to_movieId[int(i)], "?") for i in top]

def cbf_only_top(seed_movie, top_n=10):
    return similar_movies(seed_movie, top_k=top_n)["title"].tolist()

cur_top    = cur_only_top(SAMPLE_USER, 10)
cbf_top    = cbf_only_top(SEED_MOVIE, 10)
hybrid_top = hybrid_recommend(SAMPLE_USER, SEED_MOVIE, alpha=ALPHA, top_n=10)["title"].tolist()

print("CUR only (collaborative)         CBF only (content)             Hybrid (blended)")
print("-" * 100)
for a, b, c in zip(cur_top, cbf_top, hybrid_top):
    mark = "✓" if a in hybrid_top or b in hybrid_top else " "
    print(f"{a[:30]:<32} {b[:30]:<32} {c[:30]:<32}")

CUR only (collaborative)         CBF only (content)             Hybrid (blended)
----------------------------------------------------------------------------------------------------
Spirited Away (Sen to Chihiro    Delivery Man (2013)              Toy Story 3 (2010)              
Twelve Monkeys (a.k.a. 12 Monk   Maidens in Uniform (Mädchen in   Toy Story 2 (1999)              
Up (2009)                        Mechanic, The (2011)             Spirited Away (Sen to Chihiro   
Psycho (1960)                    Krampus (2015)                   Up (2009)                       
Die Hard 2 (1990)                She Wore a Yellow Ribbon (1949   Piano, The (1993)               
Piano, The (1993)                Wah-Wah (2005)                   Psycho (1960)                   
Toy Story 3 (2010)               Gummo (1997)                     Die Hard 2 (1990)               
Interstellar (2014)              Clifford (1994)                  That Thing You Do! (1996)       
Lethal Weapon 2 (1989)    

### 5.3: α sensitivity

How the blend tilts as we move from pure-content (α=0) to pure-collaborative (α=1).

In [ ]:
for alpha in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    res = hybrid_recommend(SAMPLE_USER, SEED_MOVIE, alpha=alpha, top_n=5)
    if res is None: continue
    print(f"\n--- alpha = {alpha}  (CUR weight={alpha}, CBF weight={1-alpha}) ---")
    print(res[["title", "hybrid_score"]].to_string(index=False))


--- alpha = 0.0  (CUR weight=0.0, CBF weight=1.0) ---
                  title  hybrid_score
     Toy Story 2 (1999)         1.000
     Toy Story 3 (2010)         0.992
              Up (2009)         0.549
Flintstones, The (1994)         0.531
      Annie Hall (1977)         0.529

--- alpha = 0.2  (CUR weight=0.2, CBF weight=0.8) ---
                                                                                         title  hybrid_score
                                                                            Toy Story 3 (2010)         0.930
                                                                            Toy Story 2 (1999)         0.906
                                                                                     Up (2009)         0.596
                                                                             Annie Hall (1977)         0.510
Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)         0.467

--- alp

### 5.4: Save sample hybrid output for the report

In [ ]:
SAMPLE_USERS = [1, 2, 3, 5, 10]
records = []
for uid in SAMPLE_USERS:
    res = hybrid_recommend(uid, SEED_MOVIE, alpha=ALPHA, top_n=10)
    if res is None: continue
    res["userId"] = uid
    records.append(res)

if records:
    out = pd.concat(records, ignore_index=True)
    out_path = f"{OUTPUT_DIR}/cur_hybrid_sample.csv"
    out.to_csv(out_path, index=False)
    print(f"Saved {len(out)} rows to {out_path}")
    out.head(15)

Saved 50 rows to ./cur-output/cur_hybrid_sample.csv


---
## Part 6: Summary

| Component | Value |
|---|---|
| Dataset | MovieLens 25M (or sample, see config) + TMDB 5000 |
| CUR sketch size r | 500 |
| Min support filter | 20 ratings |
| Hybrid α (default) | 0.6 |
| RMSE (test) | see Part 2.2 |
| MAE (test)  | see Part 2.2 |
| Precision@10 (filtered) | see Part 2.3 |
| Recall@10 (filtered)    | see Part 2.3 |
| HitRate@10 (test)       | see Part 2.3 |
| NDCG@10    (test)       | see Part 2.3 |
| Saved artifacts | `cur_checkpoints/`, `cur_user_recs.csv`, `cur_hybrid_sample.csv` |
